In [ ]:
import sys
import os

dust3r_path = './third_party/dust3r'
sys.path.append(dust3r_path)

sys.path.append("/home/arisrei/research/RPL/ORION/third_party/XMem")

from dust3r.inference import inference, load_model
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import os
import PIL
from PIL import Image
from tqdm import tqdm

import torch
from orion.utils.o3d_utils import *
from orion.utils.misc_utils import *
from orion.algos.xmem_tracker import XMemTracker
from orion.algos.oog import *



In [ ]:
annotation_dir = "/home/arisrei/research/RPL/ORION/annotations/human_demo/yellow_bottle"
dataset_dir = "/home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle"

In [ ]:

scene_pts_3d = np.load(os.path.join(annotation_dir,"pointcloud.npy"))
scene_colors = np.load(os.path.join(annotation_dir,"colors.npy"))

all_obj_pts = np.load(os.path.join(annotation_dir, "dust3r/all_obj_pts.npy"), allow_pickle=True)
all_obj_colors = np.load(os.path.join(annotation_dir, "dust3r/all_obj_colors.npy"), allow_pickle=True)

#extrinsics = np.load()
for idx, num in enumerate(np.linspace(0,.9,5)):
    all_obj_colors[idx][:,:] = np.array([num, num, num],dtype=np.float64)
    
#all_obj_colors[4][:,:] = np.array([.9, 0, 0],dtype=np.float64)
all_obj_points_reshaped = np.zeros((0,3))
all_obj_colors_reshaped = np.zeros((0,3))
for i in range(0,len(all_obj_pts)):
    all_obj_points_reshaped = np.concatenate((all_obj_points_reshaped, all_obj_pts[i]), axis=0)
    all_obj_colors_reshaped = np.concatenate((all_obj_colors_reshaped, all_obj_colors[i]), axis=0)


In [ ]:

pcd_raw = o3d.geometry.PointCloud()
pcd_raw.points = o3d.utility.Vector3dVector(all_obj_pts[4] + np.array([0,0,.1]))
pcd_raw.colors = o3d.utility.Vector3dVector(all_obj_colors[4])

pcd_raw_2 = o3d.geometry.PointCloud()
pcd_raw_2.points = o3d.utility.Vector3dVector(scene_pts_3d)
pcd_raw_2.colors = o3d.utility.Vector3dVector(scene_colors)

pcd_all = o3d.geometry.PointCloud()
pcd_all.points = o3d.utility.Vector3dVector(all_obj_points_reshaped)
pcd_all.colors = o3d.utility.Vector3dVector(all_obj_colors_reshaped)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = pcd_all
pcd_wrapped.max_points = 40000
print(np.array(pcd_all.points).shape, np.array(pcd_all.colors).shape)
pcd_wrapped.preprocess()
fig = visualize_o3d_point_cloud(pcd_wrapped)
fig.write_html("all_obj_pcds_no_extrinsic_transform.html")
display(fig)

In [ ]:
pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = pcd_raw_2 + pcd_all
pcd_wrapped.max_points = 40000

pcd_wrapped.preprocess()

print(np.array(pcd_wrapped.pcd.points).shape)
fig = visualize_o3d_point_cloud(pcd_wrapped)
display(fig)
fig.write_html("scene_and_obj_pcds_rescaled_with_extrinsic_transform.html")

In [ ]:
pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = pcd_raw_2
pcd_wrapped.max_points = 40000

pcd_wrapped.preprocess()

print(np.array(pcd_wrapped.pcd.points).shape)
fig = visualize_o3d_point_cloud(pcd_wrapped)
display(fig)

In [ ]:
import os
import json
import h5py
from PIL import Image
config_dict = {}
with open(os.path.join(annotation_dir, "config.json"), "r") as f:
    config_dict = json.load(f)
demo_file = h5py.File(config_dict["original_file"], "r+")
original_rgb = demo_file[f"data/human_demo/obs/agentview_rgb/"]
resized_masks = demo_file[f"data/human_demo/obs/dust3r_mask/"]
print(demo_file[f"data/human_demo/obs/"].keys())
print(original_rgb.shape)
display(Image.fromarray(original_rgb[0]))
display(Image.fromarray(resized_masks[0] * 100))

demo_file.close()
